In [1]:
import os
import json
import mne
import numpy as np
import pandas as pd
from scipy.signal import resample
from natsort import natsorted
import warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
import scipy.io as sio
import mne

In [2]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 100  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 50    # e.g. 50 means 50% overlap for SEG_LEN=100

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L100'

In [3]:
root = 'ADSZ'

In [4]:
labels = np.empty((48,2), dtype='int32')
labels.shape

(48, 2)

In [5]:
pid = 1
label_map = {'AD':1, 'Healthy':0}
for folder in os.listdir(root):
    if folder == 'Readme.txt':
        break
    folder_path = os.path.join(root,folder)
    for filename in os.listdir(folder_path):
        # print(filename)
        labels[pid-1,0] = label_map[folder]
        labels[pid-1,1] = pid
        pid += 1
labels

array([[ 1,  1],
       [ 1,  2],
       [ 1,  3],
       [ 1,  4],
       [ 1,  5],
       [ 1,  6],
       [ 1,  7],
       [ 1,  8],
       [ 1,  9],
       [ 1, 10],
       [ 1, 11],
       [ 1, 12],
       [ 1, 13],
       [ 1, 14],
       [ 1, 15],
       [ 1, 16],
       [ 1, 17],
       [ 1, 18],
       [ 1, 19],
       [ 1, 20],
       [ 1, 21],
       [ 1, 22],
       [ 1, 23],
       [ 1, 24],
       [ 0, 25],
       [ 0, 26],
       [ 0, 27],
       [ 0, 28],
       [ 0, 29],
       [ 0, 30],
       [ 0, 31],
       [ 0, 32],
       [ 0, 33],
       [ 0, 34],
       [ 0, 35],
       [ 0, 36],
       [ 0, 37],
       [ 0, 38],
       [ 0, 39],
       [ 0, 40],
       [ 0, 41],
       [ 0, 42],
       [ 0, 43],
       [ 0, 44],
       [ 0, 45],
       [ 0, 46],
       [ 0, 47],
       [ 0, 48]])

In [6]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [9]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
input_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 
                'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
n_channels = len(input_channels)


# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)
print("-------------------------------------\n")

sub = 1
for folder in os.listdir(root):
    if folder == 'Readme.txt':
        break
    folder_path = os.path.join(root,folder)
    for filename in os.listdir(folder_path):
        print("Subject", sub)
        file_path = os.path.join(folder_path, filename)
        data = []
        with open(file_path, 'r') as file:
            for line in file:
                values = [float(x) for x in line.split()]
                data.append(values)
        data_array = np.array(data)
        T_raw = data_array.shape[0]
        original_fs = 128  # original sampling rate in Hz
        for fs in SAMPLE_RATE_LIST:
            # resampled length if each trial
            T_res = int(T_raw * fs / original_fs)
            # compute number of segments
            n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
            subject_segment_counts[sub][fs] += n_seg
            N_total += n_seg
            print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
        sub += 1
        print("-------------------------------------\n")


print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 100 OVERLAP = 50 STEP = 50
-------------------------------------

Subject 1
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 2
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 3
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 4
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 5
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 6
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 7
fs=100Hz: T_res=800, STEP=50, n_seg=15
fs=50Hz: T_res=400, STEP=50, n_seg=7
-------------------------------------

Subject 8
fs=100Hz: T_res=800, STEP=50, n_seg=15


In [10]:
output_root = os.path.join('Processed', sub_folder_path, 'ADSZ')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L100\ADSZ\X.dat
y path: Processed\L100\ADSZ\y.dat


## PASS 2: Process and store segments into memmap

In [11]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
sub = 1
for folder in os.listdir(root):
    if folder == 'Readme.txt':
        break
    folder_path = os.path.join(root,folder)
    for filename in os.listdir(folder_path):
        print("Subject", sub)
        label, pid = labels[sub-1]
        file_path = os.path.join(folder_path, filename)
        data = []
        with open(file_path, 'r') as file:
            for line in file:
                values = [float(x) for x in line.split()]
                data.append(values)
        data_array = np.array(data)
        T_raw = data_array.shape[0]
        original_fs = 128  # original sampling rate in Hz
        total_seconds_all += data_array.shape[0] / original_fs
        for fs in SAMPLE_RATE_LIST:
            # resample to target fs
            data = resample_time_series(data_array, original_fs, fs)
            T_res, _ = data.shape
            print(f"fs={fs}Hz: resampled shape={data.shape}")

            # overlapped sliding window with fixed STEP (in timestamps)
            starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
            print(f"fs={fs}Hz: segments={len(starts)}")

            for s in starts:
                if cur >= N_total:
                    raise RuntimeError("Exceeded predicted N_total.")

                X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                y_mm[cur, 0] = float(label)       # label
                y_mm[cur, 1] = float(pid)         # subject ID
                y_mm[cur, 2] = float(fs)          # sample rate (scale)
                cur += 1
        sub += 1
        print("-------------------------------------\n")


total_minutes_all = total_seconds_all / 60.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total minutes across all subjects: {total_minutes_all:.2f} minutes")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Subject 1
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 2
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 3
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 4
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 5
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 6
fs=100Hz: resampled shape=(800, 19)
fs=100Hz: segments=15
fs=50Hz: resampled shape=(400, 19)
fs=50Hz: segments=7
-------------------------------------

Subject 7
fs=100Hz: resample

## Load and check the processed data

In [12]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 1128
  T = 100
  C = 19
  X_path = Processed\L100\ADSZ\X.dat
  y_path = Processed\L100\ADSZ\y.dat
-------------------------------------
Subject ID 001: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 002: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 003: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 004: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 005: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 006: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 007: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 008: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 009: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 010: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 011: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 012: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 013: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 014: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 015: X shape=(22, 100, 19), y shape=(22, 3)
Subject ID 016: X shape=(22